# Campaign Deep Agent Tool Flow Debugger

This notebook does not modify the app. It imports the same backend tools used by `/campaign/recommend` and lets you inspect the flow step by step.

Use it in two ways:

1. Run the tool cells manually to see exact inputs and outputs at each stage.
2. Run the full Deep Agent cell to see the agent-driven tool orchestration in LangSmith.

Prompt used below:

`Increase ARPU of mid-ARPU customers by 2% in 30 days.`

## 1. Environment Setup

This loads `.env`, adds `backend/` to `sys.path`, and verifies the key packages. It prints whether keys are present, but does not print secret values.

In [1]:
from pathlib import Path
import os
import sys
import json
import time
import importlib.util
from pprint import pprint

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_DIR = PROJECT_ROOT / "backend"
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / ".env", override=False)
except Exception as exc:
    print("dotenv load skipped:", exc)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", sys.executable)
for pkg in ["langchain_openai", "deepagents", "langsmith", "dotenv"]:
    print(f"{pkg:18s}", bool(importlib.util.find_spec(pkg)))

print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("LANGSMITH_API_KEY present:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("CAMPAIGN_DEEP_AGENTS_ENABLED:", os.getenv("CAMPAIGN_DEEP_AGENTS_ENABLED"))

PROJECT_ROOT: /Users/sachinpb/PycharmProjects/agents_flow/campaign_mvp
Python: /opt/homebrew/anaconda3/bin/python
langchain_openai   True
deepagents         True
langsmith          True
dotenv             True
OPENAI_API_KEY present: True
LANGSMITH_API_KEY present: True
LANGSMITH_TRACING: true
LANGSMITH_PROJECT: campaign-recommendation-mvp
CAMPAIGN_DEEP_AGENTS_ENABLED: true


## 2. Imports

These imports use the existing backend code. No source files are changed.

In [3]:
from uuid import uuid4

from app.agents.campaign_deep_agent import (
    _RUN_CONTEXTS,
    parse_objective_tool,
    lookup_rulebook_tool,
    find_segments_tool,
    load_ml_scores_tool,
    select_offer_candidates_tool,
    assemble_campaign_plan_tool,
    calculate_projection_tool,
    generate_content_tool,
    validate_campaign_tool,
    get_final_campaign_plan_tool,
    run_campaign_deep_agent_workflow,
    _create_campaign_deep_agent,
)

def as_dict(value):
    if hasattr(value, "model_dump"):
        return value.model_dump()
    if isinstance(value, dict):
        return {k: as_dict(v) for k, v in value.items()}
    if isinstance(value, list):
        return [as_dict(v) for v in value]
    return value

def pretty_json(value):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            print(value)
            return
    print(json.dumps(as_dict(value), indent=2, default=str))

def context_summary(context_id):
    ctx = _RUN_CONTEXTS[context_id]
    print("context_id:", context_id)
    print("keys:", sorted(ctx.keys()))
    if "campaign_plan" in ctx:
        plan = ctx["campaign_plan"]
        print("campaign_id:", plan.campaign_id)
        print("segments:", len(plan.recommended_segments))
        print("content_drafts:", len(plan.content_plan))
        print("has_projection:", bool(plan.projection))
        print("has_validation:", bool(plan.validation))

## 3. Create a Debug Run Context

The backend Deep Agent tools communicate through an in-memory run context. This cell creates one manually so you can call tools one by one.

In [4]:
PROMPT = "Increase ARPU of mid-ARPU customers by 2% in 30 days."
context_id = f"notebook_debug_{uuid4().hex}"

_RUN_CONTEXTS[context_id] = {
    "prompt": PROMPT,
    "preferred_campaign_type": None,
    "version": 1,
    "warnings": [],
    "errors": [],
    "messages": [],
}

context_summary(context_id)

context_id: notebook_debug_e88f8ea9c55a47b591a41a654eef334d
keys: ['errors', 'messages', 'preferred_campaign_type', 'prompt', 'version', 'warnings']


## 4. Tool 1: `parse_objective_tool`

Input to tool:

```json
{"context_id": "..."}
```

The tool reads the raw prompt from the context, calls the objective parser, and stores `parsed_objective` back into the context.

In [8]:
out = parse_objective_tool(context_id)
pretty_json(out)
context_summary(context_id)

{
  "campaign_id": "CMP_002",
  "raw_user_prompt": "Increase ARPU of mid-ARPU customers by 2% in 30 days.",
  "campaign_intent": "increase_arpu",
  "target_segment_hint": "mid_arpu",
  "target_metric": "ARPU",
  "target_lift_value": 2.0,
  "target_lift_unit": "percent",
  "time_window_value": 30,
  "time_window_unit": "days",
  "business_context": "prepaid",
  "constraints": [],
  "confidence": 0.9,
  "needs_user_clarification": false,
  "clarifying_questions": [],
  "assumptions": [
    "Objective parsed by OpenAI structured output."
  ],
  "alternative_intents": []
}
context_id: notebook_debug_e88f8ea9c55a47b591a41a654eef334d
keys: ['campaign_id', 'errors', 'messages', 'parsed_objective', 'preferred_campaign_type', 'prompt', 'version', 'warnings']


## 5. Tool 2: `lookup_rulebook_tool`

This reads `parsed_objective.campaign_intent`, then returns rulebook rows where that intent is eligible. The LLM does not invent rulebook actions; the CSV-backed tool returns them.

In [11]:
out = lookup_rulebook_tool(context_id)
pretty_json(out)
context_summary(context_id)

[
  {
    "trend": "Gradual Growth",
    "trend_meaning": "Healthy consistent growth",
    "typical_action": "Upsell / cross-sell",
    "allowed_action_families": [
      "upsell",
      "cross_sell",
      "bundle"
    ],
    "eligible_intents": [
      "increase_arpu",
      "increase_data_usage",
      "upsell",
      "cross_sell",
      "recommend_best_campaign"
    ],
    "rulebook_fit_score": 0.91
  },
  {
    "trend": "Strong Growth",
    "trend_meaning": "Strong upward revenue momentum",
    "typical_action": "Premium upsell / bundling",
    "allowed_action_families": [
      "premium_upsell",
      "bundle",
      "cross_sell"
    ],
    "eligible_intents": [
      "increase_arpu",
      "upsell",
      "cross_sell",
      "recommend_best_campaign"
    ],
    "rulebook_fit_score": 0.93
  },
  {
    "trend": "Rapid Expansion",
    "trend_meaning": "Exceptional growth",
    "typical_action": "Protect value / avoid discounts",
    "allowed_action_families": [
      "protect_value

## 6. Tool 3: `find_segments_tool`

This uses parsed objective + rulebook matches to filter mock/anonymized segment data.

In [12]:
out = find_segments_tool(context_id)
pretty_json(out)
context_summary(context_id)

[
  {
    "segment_id": "S001",
    "segment_name": "Mid ARPU Data Heavy",
    "rfm_segment": "Loyal Customers",
    "data_usage_segment": "high",
    "voice_usage_segment": "medium",
    "data_usage_trend": "Gradual Growth",
    "voice_usage_trend": "Gradual Growth",
    "customer_count": 1200000,
    "avg_arpu": 420.0,
    "avg_data_gb": 28.0,
    "avg_voice_min": 180.0,
    "recharge_frequency_days": 28,
    "churn_risk_score": 0.22,
    "activity_score": 0.81,
    "inactive_days": 0,
    "current_pack_type": "monthly",
    "offer_affinity": "data_addon",
    "business_context": "prepaid"
  },
  {
    "segment_id": "S005",
    "segment_name": "Medium Data Growth",
    "rfm_segment": "Potential Loyalists",
    "data_usage_segment": "medium",
    "voice_usage_segment": "medium",
    "data_usage_trend": "Gradual Growth",
    "voice_usage_trend": "Early Recovery",
    "customer_count": 980000,
    "avg_arpu": 310.0,
    "avg_data_gb": 15.0,
    "avg_voice_min": 140.0,
    "recharge_freq

## 7. Tool 4: `load_ml_scores_tool`

This loads mock ML outputs for selected segment IDs: channel scores, timing, expected conversion, fatigue risk, and confidence.

In [13]:
out = load_ml_scores_tool(context_id)
pretty_json(out)
context_summary(context_id)

{
  "S001": {
    "segment_id": "S001",
    "channel_scores": {
      "sms": 0.65,
      "whatsapp": 0.72,
      "push": 0.81,
      "email": 0.34,
      "ivr": 0.2,
      "outbound_call": 0.18
    },
    "best_channel": "push",
    "secondary_channel": "whatsapp",
    "best_time_window": "18:00-21:00",
    "expected_ctr": 0.18,
    "expected_conversion": 0.08,
    "fatigue_risk": "medium",
    "offer_affinity": "data_addon",
    "model_confidence": 0.83,
    "fallback_used": false,
    "fallback_reason": null
  },
  "S005": {
    "segment_id": "S005",
    "channel_scores": {
      "sms": 0.58,
      "whatsapp": 0.77,
      "push": 0.69,
      "email": 0.3,
      "ivr": 0.18,
      "outbound_call": 0.2
    },
    "best_channel": "whatsapp",
    "secondary_channel": "push",
    "best_time_window": "17:00-20:00",
    "expected_ctr": 0.16,
    "expected_conversion": 0.075,
    "fatigue_risk": "medium",
    "offer_affinity": "combo_pack",
    "model_confidence": 0.81,
    "fallback_used": 

## 8. Tool 5: `select_offer_candidates_tool`

This filters the local offer catalog by campaign intent and segment eligibility.

In [14]:
out = select_offer_candidates_tool(context_id)
pretty_json(out)
context_summary(context_id)

{
  "S001": [
    {
      "offer_id": "O001",
      "offer_name": "20GB Data Booster",
      "offer_type": "data_addon",
      "campaign_intent": "increase_arpu",
      "price": 149.0,
      "benefit": "20GB extra data",
      "validity_days": 30,
      "eligible_usage_segment": [
        "high",
        "medium"
      ],
      "eligible_rfm_segment": [
        "Loyal Customers",
        "Potential Loyalists",
        "Champions"
      ],
      "estimated_arpu_lift": 35.0,
      "estimated_data_lift_gb": 5.0,
      "estimated_save_rate": 0.0,
      "cost_per_user": 12.0,
      "margin_impact": "positive",
      "description": "Add-on pack for data-heavy users with clear incremental revenue."
    }
  ],
  "S005": [
    {
      "offer_id": "O001",
      "offer_name": "20GB Data Booster",
      "offer_type": "data_addon",
      "campaign_intent": "increase_arpu",
      "price": 149.0,
      "benefit": "20GB extra data",
      "validity_days": 30,
      "eligible_usage_segment": [
        

## 9. Tool 6: `assemble_campaign_plan_tool`

This combines parsed objective, rulebook matches, segments, ML scores, and offers into a draft `CampaignPlan` object.

In [15]:
out = assemble_campaign_plan_tool(context_id)
plan_preview = json.loads(out)
print("campaign_title:", plan_preview["campaign_title"])
print("recommended_segments:", len(plan_preview["recommended_segments"]))
print("first segment:")
pretty_json(plan_preview["recommended_segments"][0] if plan_preview["recommended_segments"] else {})
context_summary(context_id)

campaign_title: Mid ARPU Growth Campaign
recommended_segments: 3
first segment:
{
  "segment": {
    "segment_id": "S001",
    "segment_name": "Mid ARPU Data Heavy",
    "rfm_segment": "Loyal Customers",
    "data_usage_segment": "high",
    "voice_usage_segment": "medium",
    "data_usage_trend": "Gradual Growth",
    "voice_usage_trend": "Gradual Growth",
    "customer_count": 1200000,
    "avg_arpu": 420.0,
    "avg_data_gb": 28.0,
    "avg_voice_min": 180.0,
    "recharge_frequency_days": 28,
    "churn_risk_score": 0.22,
    "activity_score": 0.81,
    "inactive_days": 0,
    "current_pack_type": "monthly",
    "offer_affinity": "data_addon",
    "business_context": "prepaid"
  },
  "rulebook_match": {
    "trend": "Gradual Growth",
    "trend_meaning": "Healthy consistent growth",
    "typical_action": "Upsell / cross-sell",
    "allowed_action_families": [
      "upsell",
      "cross_sell",
      "bundle"
    ],
    "eligible_intents": [
      "increase_arpu",
      "increase_d

## 10. Tool 7: `calculate_projection_tool`

This applies deterministic formulas. For ARPU:

`incremental_revenue = eligible_users × expected_conversion × expected_arpu_lift`

In [16]:
out = calculate_projection_tool(context_id)
pretty_json(out)
context_summary(context_id)

{
  "metric": "incremental_revenue",
  "formula": "eligible_users x expected_conversion x expected_arpu_lift",
  "total_projected_impact": 6785625.0,
  "unit": "INR",
  "segment_impacts": [
    {
      "segment_id": "S001",
      "eligible_users": 1200000,
      "expected_conversion": 0.08,
      "expected_arpu_lift": 35.0,
      "projected_impact": 3360000.0
    },
    {
      "segment_id": "S005",
      "eligible_users": 980000,
      "expected_conversion": 0.075,
      "expected_arpu_lift": 35.0,
      "projected_impact": 2572500.0
    },
    {
      "segment_id": "S007",
      "eligible_users": 375000,
      "expected_conversion": 0.065,
      "expected_arpu_lift": 35.0,
      "projected_impact": 853125.0
    }
  ],
  "assumptions": [
    "Projection uses segment-level mock data, not customer-level records.",
    "Conversion, save, and lift values are MVP assumptions or mock ML outputs.",
    "Formula is deterministic and shown for auditability."
  ]
}
context_id: notebook_debug_e8

## 11. Tool 8: `generate_content_tool`

This creates draft customer-facing content. Every draft remains approval-required.

In [17]:
out = generate_content_tool(context_id)
drafts = json.loads(out)
print("draft count:", len(drafts))
pretty_json(drafts[:2])
context_summary(context_id)

draft count: 6
[
  {
    "segment_id": "S001",
    "channel": "push",
    "draft_copy": "Boost your browsing! Get 20GB extra data for just \u20b9149. Stay connected longer with our Data Booster pack valid for 30 days.",
    "tone": "direct",
    "language": "English",
    "approval_required": true,
    "approved": false,
    "why_this_copy": "This push notification is concise and highlights the key offer benefits tailored to high data users in the loyal customer segment, encouraging incremental ARPU through a data add-on.",
    "compliance_notes": [
      "No guaranteed savings or misleading claims included.",
      "Price and validity clearly stated."
    ]
  },
  {
    "segment_id": "S001",
    "channel": "whatsapp",
    "draft_copy": "Hi! Noticed you love using data. How about adding 20GB extra data for \u20b9149? It\u2019s valid for 30 days and perfect if you want to keep streaming, browsing, and chatting without worries. Interested? Just reply to know more!",
    "tone": "conversa

## 12. Tool 9: `validate_campaign_tool`

This checks rulebook compliance, projection compliance, channel support, content draft state, frequency/quiet-hour guardrails, and export readiness.

In [18]:
out = validate_campaign_tool(context_id)
pretty_json(out)
context_summary(context_id)

{
  "is_valid": true,
  "warnings": [
    "Uses mock data.",
    "Uses mock ML scores.",
    "Projection is simplified.",
    "Content requires approval.",
    "No production campaign launch integration."
  ],
  "errors": [],
  "rulebook_compliance": "passed",
  "projection_compliance": "passed",
  "content_compliance": "passed"
}
context_id: notebook_debug_e88f8ea9c55a47b591a41a654eef334d
keys: ['campaign_id', 'campaign_plan', 'content_plan', 'errors', 'messages', 'ml_scores', 'offer_candidates', 'parsed_objective', 'preferred_campaign_type', 'projection', 'prompt', 'rulebook_matches', 'segment_candidates', 'selected_segments', 'validation_result', 'version', 'warnings']
campaign_id: CMP_002
segments: 3
content_drafts: 6
has_projection: True
has_validation: True


## 13. Tool 10: `get_final_campaign_plan_tool`

This returns the final typed campaign plan stored in the context.

In [19]:
out = get_final_campaign_plan_tool(context_id)
final_plan = json.loads(out)
print("campaign_id:", final_plan["campaign_id"])
print("title:", final_plan["campaign_title"])
print("summary:", final_plan["summary"])
print("valid:", final_plan["validation"]["is_valid"])
print("warnings:")
pprint(final_plan["validation"]["warnings"])

campaign_id: CMP_002
title: Mid ARPU Growth Campaign
summary: This campaign targets mid-ARPU customers to increase their ARPU by 2% within 30 days through upsell and cross-sell offers. The primary offer is the '20GB Data Booster' tailored to different segments with optimized channels and timing to maximize conversion and confidence. The segments include high data usage loyal customers, medium data growth potential loyalists, and voice-first loyalists with low data usage, each receiving personalized communication via push, WhatsApp, or SMS respectively.
valid: True
warnings:
['Uses mock data.',
 'Uses mock ML scores.',
 'Projection is simplified.',
 'Content requires approval.',
 'No production campaign launch integration.']


## 14. Full Deep Agent Run

This calls the same Deep Agent workflow used by FastAPI. In LangSmith, look for a root run named `campaign_deep_agent_orchestrator`. Under it, you should see tool calls such as `parse_objective_tool`, `lookup_rulebook_tool`, and `validate_campaign_tool`.

This cell may take longer because it calls the model and lets the agent decide tool use.

In [20]:
start = time.time()
state = run_campaign_deep_agent_workflow(PROMPT)
elapsed = time.time() - start

plan = state["campaign_plan"]
print(f"elapsed: {elapsed:.2f}s")
print("campaign_id:", plan.campaign_id)
print("segments:", len(plan.recommended_segments))
print("valid:", plan.validation.is_valid if plan.validation else None)
print("warnings:")
pprint(state["warnings"])
print("errors:")
pprint(state["errors"])

elapsed: 36.48s
campaign_id: CMP_004
segments: 3
valid: True
warnings:
['Objective parsed by OpenAI structured output.',
 'Uses mock data.',
 'Uses mock ML scores.',
 'Projection is simplified.',
 'Content requires approval.',
 'No production campaign launch integration.']
errors:
[]


## 15. Optional: Stream Deep Agent Events Locally

This lower-level cell creates a fresh Deep Agent and streams chunks locally. LangSmith is still the better UI for the full tree, but this helps you see progress in the notebook.

In [21]:
stream_context_id = f"notebook_stream_{uuid4().hex}"
_RUN_CONTEXTS[stream_context_id] = {
    "prompt": PROMPT,
    "preferred_campaign_type": None,
    "version": 1,
    "warnings": [],
    "errors": [],
    "messages": [],
}

agent = _create_campaign_deep_agent()
inputs = {
    "messages": [
        {
            "role": "user",
            "content": (
                f"Run a campaign recommendation for context_id={stream_context_id}. "
                f"User objective: {PROMPT}. Use the campaign tools and return a short final note."
            ),
        }
    ]
}

for i, chunk in enumerate(agent.stream(inputs, config={"run_name": "notebook_campaign_deep_agent_stream"})):
    print("--- chunk", i, "---")
    pprint(chunk)
    if i >= 20:
        print("Stopping display after 20 chunks; the agent may continue if you remove this break.")
        break

--- chunk 0 ---
{'SummarizationMiddleware.before_model': None}
--- chunk 1 ---
{'model_request': {'messages': [AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_yoCxDFi4tpks6KDnGVH6QR3R', 'function': {'arguments': '{"context_id":"notebook_stream_2b89911a424f43b299b56d123abaaaf1"}', 'name': 'parse_objective_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 6070, 'total_tokens': 6107, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 5888}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_2b1324456b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--2520202b-1915-4e00-9894-337f9b6f3f60-0', tool_calls=[{'name': 'parse_objective_tool', 'args': {'context_id': 'notebook_stream_2b89911a424f43b299b56d123abaaaf1'}, 'id': 'cal